# Quoridor Chase: Lockdown
## Bagian 2.1: Algoritma Minimax & Alpha-Beta Pruning
**Mata Kuliah:** Artificial Intelligence (Semester 5) — Universitas Ciputra  
**Kelompok:** Grup BESAR  
**Penanggung Jawab (PIC):**  
- **Amadeus Ian Gunadi (Deus)** — `0706012410051`  
- **Muhammad Althaf Hilmi (Althaf)** — `0706012410065`  

---
Notebook ini mendemonstrasikan algoritma **Minimax** dan optimasi **Alpha-Beta Pruning** untuk pengambilan keputusan AI Boss (Node MAX) melawan Tim Runners (Node MIN).

Terdapat **2 Mode** yang dapat diaktifkan dan dinonaktifkan melalui tombol toggle di bawah ini:
1. **Alpha-Beta Pruning (ON - Direkomendasikan):** Memangkas cabang pohon pencarian yang terbukti lebih buruk dari batas $\alpha$ dan $\beta$.
2. **Pure Minimax (OFF - Baseline):** Menelusuri seluruh pohon pencarian secara lengkap sebagai pembanding performa (*ablation study*).

### 1. Kontrol Tombol Mode Pencarian (Alpha-Beta Toggle)
Gunakan tombol di bawah untuk memilih mode pencarian AI Boss.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

from engine.state import GameState
from src.ai.minimax import MinimaxAgent

# Inisialisasi Game State dan AI Agent
state = GameState.create_initial_state(seed=42)
agent = MinimaxAgent(depth_limit=2, use_alpha_beta=True)

try:
    import ipywidgets as widgets
    from IPython.display import display

    btn_toggle = widgets.ToggleButtons(
        options=['Alpha-Beta Pruning (ON)', 'Pure Minimax (OFF)'],
        description='Mode AI:',
        disabled=False,
        button_style='success', # 'success', 'info', 'warning', 'danger' or ''
        tooltips=['Aktifkan pemangkasan Alpha-Beta (Cepat & Optimal)', 'Minimax Murni tanpa pemangkasan (Baseline)'],
        icons=['bolt', 'hourglass-half']
    )

    def on_mode_change(change):
        is_ab = (change['new'] == 'Alpha-Beta Pruning (ON)')
        agent.set_alpha_beta_mode(is_ab)
        print(f"\n[STATUS] Mode diubah menjadi: {agent.mode_name}")

    btn_toggle.observe(on_mode_change, names='value')
    display(btn_toggle)
except ImportError:
    print("[INFO] ipywidgets tidak terinstall. Mode default: Alpha-Beta Pruning (ON).")
    print("Gunakan `agent.set_alpha_beta_mode(False)` untuk mematikan Alpha-Beta Pruning.")

### 2. Eksekusi Pengambilan Keputusan AI Boss
Menjalankan pencarian adversarial pada kedalaman yang dipilih dan mencatat metrik pencarian.

In [ ]:
# Eksekusi pemilihan langkah terbaik
best_action, eval_score = agent.choose_action(state, depth=2)
metrics = agent.metrics.to_dict()

print("=" * 50)
print(f"Mode Pencarian        : {agent.mode_name}")
print(f"Langkah Terbaik (Best): {best_action}")
print(f"Nilai Evaluasi h(s)   : {eval_score:.2f}")
print(f"Node Dieksplorasi     : {metrics['nodes_explored']} node")
print(f"Jumlah Pruning Cutoffs: {metrics['cutoffs']}")
print(f"Waktu Eksekusi        : {metrics['execution_time_ms']} ms")
print("=" * 50)

### 3. Eksperimen Perbandingan: Pure Minimax vs Alpha-Beta Pruning
Sesuai dengan **Proposal Bab 2.3.d (Hipotesis)**, Alpha-Beta Pruning diharapkan memangkas $> 50\%$ node yang dievaluasi dengan hasil keputusan yang tetap optimal.

In [ ]:
# Evaluasi State pada Depth 2 untuk kedua mode
agent_pure = MinimaxAgent(depth_limit=2, use_alpha_beta=False)
act_pure, val_pure = agent_pure.choose_action(state)
m_pure = agent_pure.metrics.to_dict()

agent_ab = MinimaxAgent(depth_limit=2, use_alpha_beta=True)
act_ab, val_ab = agent_ab.choose_action(state)
m_ab = agent_ab.metrics.to_dict()

reduction = (1.0 - (m_ab['nodes_explored'] / m_pure['nodes_explored'])) * 100.0

print(f"{'Metrik Pengujian':<25} | {'Pure Minimax':<18} | {'Alpha-Beta Pruning':<18}")
print("-" * 68)
print(f"{'Langkah Terbaik':<25} | {str(act_pure):<18} | {str(act_ab):<18}")
print(f"{'Nilai Evaluasi h(s)':<25} | {val_pure:<18.2f} | {val_ab:<18.2f}")
print(f"{'Node Dieksplorasi':<25} | {m_pure['nodes_explored']:<18} | {m_ab['nodes_explored']:<18}")
print(f"{'Pruning Cutoffs':<25} | {m_pure['cutoffs']:<18} | {m_ab['cutoffs']:<18}")
print(f"{'Waktu Eksekusi (ms)':<25} | {m_pure['execution_time_ms']:<18.2f} | {m_ab['execution_time_ms']:<18.2f}")
print("-" * 68)
print(f"[HASIL] Efisiensi Pemangkasan Node: {reduction:.1f}% pemangkasan!")
print(f"[HASIL] Kesetaraan Keputusan (Optimality): {'LULUS (Identik)' if abs(val_pure - val_ab) < 1e-4 else 'GAGAL'}")